# Notebook for Training the MacroTrader Models

In [1]:
# installing packages for you
!pip install --upgrade setuptools packaging
!pip install .

Processing /home/ec2-user/SageMaker/Buy_side_model/Blockhouse-ML
  Preparing metadata (setup.py) ... done
  Created wheel for blockhouse-ml: filename=blockhouse_ml-0.1.0-py3-none-any.whl size=43543 sha256=8b46d2ea74dc6959849301c2f732a70d41c6e4fc73ba7289732e71c24400b6f6
  Stored in directory: /home/ec2-user/.cache/pip/wheels/7d/3c/79/57376f093a4956b5846d17a012a11f18d8e621b854f7dfb16f
Successfully built blockhouse-ml
  Attempting uninstall: blockhouse-ml
    Found existing installation: blockhouse-ml 0.1.0
    Uninstalling blockhouse-ml-0.1.0:
      Successfully uninstalled blockhouse-ml-0.1.0


In [4]:
# Install TA-Lib using conda
!conda install -c conda-forge ta-lib -y

Retrieving notices: ...working... done
Solving environment: | 
The environment is inconsistent, please check the package plan carefully
The following packages are causing the inconsistenc/ 

  - conda-forge/noarch::autopep8==2.0.4=pyhd8ed1ab_0
  - conda-forge/linux-64::black==24.4.2=py310hff52083_0
  - conda-forge/noarch::bleach==6.1.0=pyhd8ed1ab_0
  - conda-forge/noarch::plotly==5.23.0=pyhd8ed1ab_0
  - conda-forge/noarch::pytest==8.3.2=pyhd8ed1ab_0
  - conda-forge/noarch::qtpy==2.4.1=pyhd8ed1ab_0
  - conda-forge/linux-64::sip==6.7.12=py310hc6cd4ac_0
  - conda-forge/noarch::flask==3.0.3=pyhd8ed1ab_0
  - conda-forge/noarch::importlib_metadata==8.2.0=hd8ed1ab_0
  - conda-forge/noarch::lazy_loader==0.4=pyhd8ed1ab_0
  - conda-forge/linux-64::pyqt5-sip==12.12.2=py310hc6cd4ac_5
  - conda-forge/noarch::pytoolconfig==1.2.5=pyhd8ed1ab_0
  - conda-forge/noarch::qdarkstyle==3.1=pyhd8ed1ab_0
  - conda-forge/noarch::qtawesome==1.3.1=pyh9208f05_0
  - conda-forge/noarch::yapf==0.40.1=pyhd8ed1ab_0
  -

In [ ]:
!nvidia-smi

In [6]:
!pip install databento

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.3/79.3 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 83.3 MB/s eta 0:00:00:00:010:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.22.4
    Uninstalling numpy-1.22.4:
      Successfully uninstalled numpy-1.22.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mkl-fft 1.3.10 requires mkl, which is not installed.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.1.1 which is incompatible.
hdijupyterutils 0.21.0 requires pandas<2.0.0,>=0.17.1, but you have pandas 2.2.2 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.1.1 which is incompatible.
pytorch-widede

In [1]:
from blockhouse_ml.utils.fetch_merge_data import PolygonClient
from blockhouse_ml.utils.fetch_merge_data import *
from blockhouse_ml.utils.data_handler import DataProcessor
from blockhouse_ml.utils.macro_model_utils import MacroTraderModel
from blockhouse_ml.utils.macro_model import MetaLearner
import os

ModuleNotFoundError: No module named 'polygon'

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from stable_baselines3 import PPO
from gym import spaces

In [ ]:
# Initialize the model directory, where the models will be saved
UNET_MODEL_DIR = 'Models'
os.makedirs(UNET_MODEL_DIR, exist_ok=True)

TAB_MODEL_DIR = 'TabModels'
os.makedirs(TAB_MODEL_DIR, exist_ok=True)

# Initialize the data directory, where the data will be stored
data_dir = 'Data'
os.makedirs(data_dir, exist_ok=True)

# Initialize the log
log_dir = 'Logs'

# Initialize the MetaLearner
metalearner = MetaLearner()

# Initialize the MacroTraderModel
macro_trader_unet = MacroTraderModel(UNET_MODEL_DIR)

macro_trader_tab = MacroTraderModel(TAB_MODEL_DIR)


# Initialize the Date Client
data_client = PolygonClient(save_dir=data_dir)

# Initialize the data processor
data_processor = DataProcessor()

## Model Training parameters
training_params = { 'callback' : None, 'total_timesteps' : 10000}

# Define the forecast steps
forecast_steps = {
    'open': (360, '1T'),
    'high': (360, '1T'),
    'low': (360, '1T'),
    'close': (360, '1T'),
    'volatility': (360, '1T'),
    'volume': (360, '1T'),
    'transaction_cost': (360, '1T')
}

# Define the start and end dates
start_time = '2024-07-01'
end_time = '2024-07-10'

# List of Companies based on Market Capitalization
large_cap_companies = ['AAPL', 'CSCO', 'MCD','IBM', 'AMZN', 'TSLA', 'PFE', 'MS','MSFT','NVDA']
# mid_cap_companies = ['AEG', 'NICE', 'NLY', 'ONTO', 'PSN', 'SAIA', 'OWL','PNW','TWLO','HAS']
# small_cap_companies = ['NVAX','AMC','WOLF','IREN','SEDG', 'UPWK','SERV','FSLY','BMBL','ARRY']


# Fetch and process the data for Training

In [ ]:
def get_data(data_dir, ticker, start_time, end_time):
     ## Fetch and process the data and save it to the data directory, if it doesn't exist
    processed_filename = f'{data_dir}/processed_data_{ticker}_{start_time}_{end_time}.csv'
    if os.path.exists(processed_filename):
        processed_data = pd.read_csv(processed_filename)
    else:
        filename = f'{data_dir}/merged_data_{ticker}_{start_time}_{end_time}.csv'
        if os.path.exists(filename):
            data = pd.read_csv(filename)
        else:
            data = data_client.fetch_and_merge_data(ticker,start_date=start_time,end_date=end_time)
            data.to_csv(filename)

        processed_data = data_processor.process_data(data, forecast_steps,n_jobs=4)
        processed_data.to_csv(processed_filename)

    return processed_data

In [ ]:
from collections import defaultdict

# Initialize a dictionary to store the results
results = defaultdict(list)
data_dir = "./Data/"
# Iterate over the lists of companies (large, mid, small cap)
large_cap_companies = ['AAPL', 'CSCO', 'MCD','IBM', 'AMZN', 'TSLA', 'PFE', 'MS','MSFT','NVDA']

for companies in [large_cap_companies]:#, mid_cap_companies, small_cap_companies]:

    for ticker in companies:
        
        file_name = f"processed_data_{ticker}_2024-07-01_2024-07-10.csv"
        file_path = os.path.join(data_dir, file_name)
        if os.path.exists(file_path):
            # Load the data directly from the CSV file
            processed_data = pd.read_csv(file_path)
            processed_data['ticker'] = ticker
            train_data = processed_data[:int(len(processed_data) * 0.8)]
            test_data = processed_data[int(len(processed_data) * 0.8):]

            print("Training Data Shape: ", train_data.shape)
            print("Training Model for ", ticker)
        
            # Fetch the market cap for the ticker
            market_cap_int = get_market_cap(ticker)
            market_cap = metalearner.classify_market_cap(market_cap_int)

            # Train and test for different transaction sizes
            for transaction_size in [ 10000]:#9, 99, 499, 1999,
                scenario = metalearner.classify_scenario(transaction_size)
                print("Training Model for ticker: {} Transaction Size: {} Market Cap: {}".format(ticker, scenario, market_cap))

                # Set the logging parameters
                training_params['tb_log_name'] = f'{log_dir}/{ticker}_{scenario}_{market_cap}'

                # Train the model with the TabTransformer option enabled
                model, _ = macro_trader_tab.train(train_data, market_cap=market_cap, scenario=scenario, resume=True, training_params=training_params, get_tab_transformer=True)

                print("------------------------")
                print("Testing Model for ticker: {} Transaction Size: {} Market Cap: {}".format(ticker, scenario, market_cap))

                # Test the model
                rew, _ = macro_trader_tab.test(test_data, model=model, market_cap=market_cap, scenario=scenario)
               
                # Store the results
                results[ticker].append((market_cap, scenario, rew))
                print("Reward: ", rew)
                print("------------------------")


Training Data Shape:  (2128, 54)
Training Model for  AAPL
Training Model for ticker: AAPL Transaction Size: large Market Cap: large
large cap, large scenario model selected


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/gym/spaces/box.py:127: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pytorch_widedeep/utils/general_utils.py:12: DeprecationWarning: The 'embed_continuous' parameter is deprecated and will be removed in the next release. Please use 'embed_continuous_method' instead See the documentation for more details.
  return func(*args, **kwargs)


Model not found at TabModels/model_large_cap_large.h5, training from scratch...
------------------------
Testing Model for ticker: AAPL Transaction Size: large Market Cap: large
large cap, large scenario model selected


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/gym/spaces/box.py:127: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


Reward:  -2142527.5202401164
------------------------
Training Data Shape:  (2032, 54)
Training Model for  CSCO
Training Model for ticker: CSCO Transaction Size: large Market Cap: large
large cap, large scenario model selected
Model loaded from TabModels/model_large_cap_large.h5


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/gym/spaces/box.py:127: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pytorch_widedeep/utils/general_utils.py:12: DeprecationWarning: The 'embed_continuous' parameter is deprecated and will be removed in the next release. Please use 'embed_continuous_method' instead See the documentation for more details.
  return func(*args, **kwargs)
/home/ec2-user/SageMaker/Buy_side_model/Blockhouse-ML/blockhouse_ml/utils/macro_model_utils.py:396: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for mo

KeyboardInterrupt: 

In [12]:
!conda list numpy scipy gensim


usage: conda [-h] [-V] command ...
conda: error: unrecognized arguments: scipy gensim

Note: you may need to restart the kernel to use updated packages.


In [11]:
# Fetch and process the data and save it to the data directory
for companies in [large_cap_companies]:#, mid_cap_companies, small_cap_companies]:

    for ticker in companies:
        
        processed_data = get_data(data_dir, ticker, start_time, end_time)
        

        

/home/ec2-user/SageMaker/Buy_side_model/Blockhouse-ML/blockhouse_ml/utils/fetch_merge_data.py:348: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  result = df.resample('1T').agg(agg_functions)
Processing rows: 4it [00:00, 18.01it/s]/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/gensim/utils.py:35: UserWarning: A NumPy version >=1.22.4 and <1.29.0 is required for this version of SciPy (detected version 2.1.1)
  import scipy.sparse
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/gensim/utils.py:35: UserWarning: A NumPy version >=1.22.4 and <1.29.0 is required for this version of SciPy (detected version 2.1.1)
  import scipy.sparse

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of th

BrokenProcessPool: A task has failed to un-serialize. Please ensure that the arguments of the function are all picklable.

Processing rows: 4it [00:19, 18.01it/s]

In [ ]:
import os
import pandas as pd
from collections import defaultdict
# import os

current_directory = os.getcwd()
print("Current working directory:", current_directory)

results = defaultdict(list)

# Assuming the files are named like 'processed_data_<TICKER>_<START_DATE>_<END_DATE>.csv'
# data_dir = "../../prasana_ml/Blockhouse-ML/Data"
data_dir = "./Data"

for companies in [large_cap_companies]:#, mid_cap_companies, small_cap_companies]:
    for ticker in companies:
        # Construct the expected filename
        file_name = f"processed_data_{ticker}_2024-07-01_2024-07-10.csv"
        file_path = os.path.join(data_dir, file_name)

        print(f"Looking for file: {file_path}")

        if os.path.exists(file_path):
            # Load the data directly from the CSV file
            processed_data = pd.read_csv(file_path)
            processed_data['ticker'] = ticker
            train_data = processed_data[:int(len(processed_data) * 0.8)]
            test_data = processed_data[int(len(processed_data) * 0.8):]

            print(f"Training Data Shape: {train_data.shape}")

            print("Training Model for", ticker)
            market_cap_int = get_market_cap(ticker)
            market_cap = metalearner.classify_market_cap(market_cap_int)

            for transaction_size in [10000]:
                scenario = metalearner.classify_scenario(transaction_size)
                print(f"Training Model for ticker: {ticker} Transaction Size: {scenario} Market Cap: {market_cap}")

                training_params['tb_log_name'] = f'{log_dir}/{ticker}_{scenario}_{market_cap}'
                model, _ = macro_trader_unet.train(train_data, market_cap=market_cap, scenario=scenario, resume=True, training_params=training_params)
                print("------------------------")
                print(f"Testing Model for ticker: {ticker} Transaction Size: {scenario} Market Cap: {market_cap}")

                rew, _ = macro_trader_unet.test(test_data, model=model, market_cap=market_cap, scenario=scenario)
                results[ticker].append((market_cap, scenario, rew))
                print("Reward:", rew)
                print("------------------------")
        else:
            print(f"Data file not found for {ticker}. Please ensure the file {file_name} exists.")


# Training the Unet-Transformer MacroTrader Model without Fine Tuning 

In [ ]:
from collections import defaultdict
results = defaultdict(list)

for companies in [large_cap_companies]:#, mid_cap_companies, small_cap_companies]:

    for ticker in companies:

        processed_data = get_data(data_dir, ticker, start_time, end_time)
        
        train_data = processed_data[:int(len(processed_data) * 0.8)]
        test_data = processed_data[int(len(processed_data) * 0.8):]

        print (" Training Data Shape: ", train_data.shape)

        print("Training Model for ", ticker)
        market_cap_int = fetch_merge_data.get_market_cap(ticker)
        market_cap = metalearner.classify_market_cap(market_cap_int)
        for transaction_size in [9, 99, 499, 1999, 10000]:
            scenerio = metalearner.classify_scenario(transaction_size)
            print("Training Model for ticker: {} Transaction Size: {} Market Cap: {}".format(ticker, scenerio, market_cap))
            training_params['tb_log_name'] =f'{log_dir}/{ticker}_{scenerio}_{market_cap}'
            model, _ = macro_trader_unet.train(train_data, market_cap=market_cap,scenario=scenerio,resume=True,training_params = training_params)
            print("------------------------")
            print("Testing Model for ticker: {} Transaction Size: {} Market Cap: {}".format(ticker, scenerio, market_cap))
            rew, _ = macro_trader_unet.test(test_data, model=model, market_cap=market_cap, scenario=scenerio)
            results[ticker].append((market_cap, scenerio, rew))
            print("Reward: ", rew)
            print("------------------------")


# Training the TAB-Transformer MacroTrader Model without Fine Tuning 

In [ ]:
from collections import defaultdict
results = defaultdict(list)

for companies in [large_cap_companies, mid_cap_companies, small_cap_companies]:

    for ticker in companies:
        
        processed_data = get_data(data_dir, ticker, start_time, end_time)
        
        train_data = processed_data[:int(len(processed_data) * 0.8)]
        test_data = processed_data[int(len(processed_data) * 0.8):]

        print (" Training Data Shape: ", train_data.shape)

        print("Training Model for ", ticker)
        market_cap_int = fetch_merge_data.get_market_cap(ticker)
        market_cap = metalearner.classify_market_cap(market_cap_int)
        for transaction_size in [9, 99, 499, 1999, 10000]:
            scenerio = metalearner.classify_scenario(transaction_size)
            print("Training Model for ticker: {} Transaction Size: {} Market Cap: {}".format(ticker, scenerio, market_cap))
            training_params['tb_log_name'] =f'{log_dir}/{ticker}_{scenerio}_{market_cap}'
            model, _ = macro_trader_tab.train(train_data, market_cap=market_cap,scenario=scenerio,resume=True,training_params = training_params, get_tab_transformer=True)
            print("------------------------")
            print("Testing Model for ticker: {} Transaction Size: {} Market Cap: {}".format(ticker, scenerio, market_cap))
            rew, _ = macro_trader_tab.test(test_data, model=model, market_cap=market_cap, scenario=scenerio)
            results[ticker].append((market_cap, scenerio, rew))
            print("Reward: ", rew)
            print("------------------------")


# Training with finetuning

In [2]:
import ray
import os
cwd = os.getcwd()
print(cwd)
path = "/home/ec2-user/SageMaker/"
ray.init(local_mode=True, num_gpus=1, num_cpus=16, dashboard_host='0.0.0.0', log_to_driver=False,
             _temp_dir=os.path.join(path, "ray_results_buy"), ignore_reinit_error=True)
# ray.init(local_mode=True, num_gpus=1, num_cpus=16, dashboard_host='0.0.0.0', log_to_driver=False,
#          _temp_dir="/tmp/ray_results", ignore_reinit_error=True)


/home/ec2-user/SageMaker/Buy_side_model/Blockhouse-ML


2024-09-03 21:01:12,151	INFO worker.py:1783 -- Started a local Ray instance.


Python version:,3.10.14
Ray version:,2.35.0


# Declare the variables for finetuning it can be different then training the model above

In [7]:
# Initialize the model directory, where the models will be saved
MODEL_DIR = 'Fine-Tuned-Models'
os.makedirs(MODEL_DIR, exist_ok=True)

# Initialize the data directory, where the data will be stored
data_dir = 'Data'
os.makedirs(data_dir, exist_ok=True)

# Initialize the log
log_dir = 'Logs'

# Initialize the MetaLearner
metalearner = MetaLearner()

# Initialize the MacroTraderModel
macro_trader_unet = MacroTraderModel(MODEL_DIR)
macro_trader_tab = MacroTraderModel(TAB_MODEL_DIR)
# Initialize the data processor
data_processor = DataProcessor()

## Model Training parameters
training_params = { 'callback' : None, 'total_timesteps' : 10000}

# Define the forecast steps
forecast_steps = {
    'open': (360, '1T'),
    'high': (360, '1T'),
    'low': (360, '1T'),
    'close': (360, '1T'),
    'volatility': (360, '1T'),
    'volume': (360, '1T'),
    'transaction_cost': (360, '1T')
}

# Define the start and end dates
start_time = '2024-07-01'
end_time = '2024-07-10'

# List of Companies based on Market Capitalization, for fine tuning we only use 1 company for each market cap
large_cap_companies = ['AAPL']
# mid_cap_companies = ['AEG']
# small_cap_companies = ['NVAX']


In [15]:
from ray import tune, train
from ray.tune.schedulers import ASHAScheduler
from ray.tune.search.hyperopt import HyperOptSearch
import ray
import os

current_directory = os.getcwd()
print("Current working directory:", current_directory)
new_directory =  "/home/ec2-user/SageMaker/Buy_side_model/Blockhouse-ML/"
os.chdir(new_directory)
storage_path = "~/tune_results_buy"

# Training Config
"""
Key required for fine tuning
learning_rate, n_steps, batch_size, gamma, clip_range, n_epochs, ent_coef, resume, total_timesteps, callback, tb_log_name, train_data, market_cap, scenario, test_data
"""
data_dir = "./Data"
finetune_config = {
    "learning_rate": tune.loguniform(1e-5, 1e-1),
    "n_steps": tune.choice([128, 256, 512]),
    "batch_size": tune.choice([64, 128, 256]),
    "gamma": tune.uniform(0.9, 0.999),
    "clip_range" : tune.uniform(0.1, 0.4),
    "n_epochs": tune.choice([4, 6, 8]),
    "ent_coef": tune.loguniform(0.0001, 0.1),
    "resume" : False,
    "total_timesteps": training_params['total_timesteps'],
    "callback" : training_params['callback']
}

scheduler = ASHAScheduler(
    metric="reward",
    mode="max",
)

# hyperopt_search = HyperOptSearch(finetune_config, metric="reward", mode="max")


for companies in [large_cap_companies]:#, mid_cap_companies, small_cap_companies]:

    for ticker in companies:
#         filename = f'{data_dir}/merged_data_{ticker}_{start_time}_{end_time}.csv'
#         print(filename)
#         if os.path.exists(filename):
#             data = pd.read_csv(filename)
#         else:
#             print("data not found")
#             data = fetch_and_merge_data(ticker,start_date=start_time,end_date=end_time,save_dir =data_dir)

#         processed_filename = f'{data_dir}/processed_data_{ticker}_{start_time}_{end_time}.csv'

#         if os.path.exists(processed_filename):
#             processed_data = pd.read_csv(processed_filename)
#         else:
#             processed_data = data_processor.process_data(data, forecast_steps)
#             processed_data.to_csv(processed_filename)
        # processed_data_path = "Buy_side_model/Blockhouse-ML/Data/processed_data_AAPL_2024-07-01_2024-07-10.csv"
        # processed_data = pd.read_csv(processed_data_path)
        train_data = processed_data[:int(len(processed_data) * 0.8)]
        test_data = processed_data[int(len(processed_data) * 0.8):]

        # print (" Training Data Shape: ", train_data.shape)

        # print("Training Model for ", ticker)
        market_cap_int = get_market_cap(ticker)
        market_cap = metalearner.classify_market_cap(market_cap_int)
        for transaction_size in [1000]:#9, 99, 499, 1999, 10000]:
            scenerio = metalearner.classify_scenario(transaction_size)

            finetune_config["train_data"] = train_data
            finetune_config["test_data"] = test_data
            finetune_config["market_cap"] = market_cap
            finetune_config["scenario"] = scenerio
            
            analysis = tune.Tuner(
                macro_trader_tab.fine_tuning_model,
                param_space=finetune_config,
                run_config=train.RunConfig(
                    name=f"{ticker}_{scenerio}_{market_cap}",
                    storage_path="~/tune_results",
                ),
                tune_config=tune.TuneConfig(
                    scheduler=scheduler,
                     trial_dirname_creator=lambda trial: trial.trial_id)
            )

            results = analysis.fit()
            print(results.get_dataframe())
            print(results.metrics_dataframe())
            break
        break

# print("Best config: ", analysis.get_best_config(metric="mean_reward", mode="max"))
# df = analysis.results_df
# print(df.head())

:job_id:01000000
:task_name:bundle_reservation_check_func
:actor_name:ImplicitFunc
:actor_name:fine_tuning_model
large cap, large scenario model selected
Model not found at TabModels/model_large_cap_large.h5, training from scratch...


:job_id:01000000
:task_name:bundle_reservation_check_func
:actor_name:ImplicitFunc
:actor_name:fine_tuning_model
2024-09-03 20:55:26,415	ERROR tune_controller.py:1331 -- Trial task failed for trial fine_tuning_model_d8420_00000
Traceback (most recent call last):
  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/ray/air/execution/_internal/event_manager.py", line 110, in resolve_future
    result = ray.get(future)
  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/ray/_private/auto_init_hook.py", line 21, in auto_init_wrapper
    return fn(*args, **kwargs)
  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/ray/_private/client_mode_hook.py", line 103, in wrapper
    return func(*args, **kwargs)
  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/ray/_private/worker.py", line 2661, in get
    values, debugger_breakpoint = worker.get_objects(object_refs, timeout=timeout)
  File "/home/ec2-user/a

<IPython.core.display.HTML object>
<IPython.core.display.HTML object>
Empty DataFrame
Columns: []
Index: []


AttributeError: 'ResultGrid' object has no attribute 'metrics_dataframe'

In [10]:
import os
import ray
import pandas as pd
from ray import tune, train
from ray.tune.schedulers import ASHAScheduler


import os

current_directory = os.getcwd()
new_directory =  "/home/ec2-user/SageMaker/Buy_side_model/Blockhouse-ML/"
os.chdir(new_directory)
print("Current working directory:", current_directory)
# Define training parameters
training_params = {
    'callback': None,
    'total_timesteps': 10000
}

# Define the forecast steps
forecast_steps = {
    'open': (360, '1T'),
    'high': (360, '1T'),
    'low': (360, '1T'),
    'close': (360, '1T'),
    'volatility': (360, '1T'),
    'volume': (360, '1T'),
    'transaction_cost': (360, '1T')
}

# Set the start and end dates
start_time = '2024-07-01'
end_time = '2024-07-10'

# Define the path for the processed data
processed_data_path = "./Data/processed_data_AAPL_2024-07-01_2024-07-10.csv"

# Check if the processed data file exists and load it
try:
    processed_data = pd.read_csv(processed_data_path)
    train_data = processed_data[:int(len(processed_data) * 0.8)]
    test_data = processed_data[int(len(processed_data) * 0.8):]
except FileNotFoundError:
    print(f"File not found: {processed_data_path}")
    # Handle the case where the file doesn't exist

# Set up Ray
ray.init(local_mode=True, num_gpus=1, num_cpus=16, dashboard_host='0.0.0.0', log_to_driver=False,
         _temp_dir="/tmp/ray_results", ignore_reinit_error=True)

# Example large cap companies list
large_cap_companies = ['AAPL']

# Fine-tuning configuration
finetune_config = {
    "learning_rate": tune.loguniform(1e-5, 1e-1),
    "n_steps": tune.choice([128, 256, 512]),
    "batch_size": tune.choice([64, 128, 256]),
    "gamma": tune.uniform(0.9, 0.999),
    "clip_range": tune.uniform(0.1, 0.4),
    "n_epochs": tune.choice([4, 6, 8]),
    "ent_coef": tune.loguniform(0.0001, 0.1),
    "resume": False,
    "total_timesteps": training_params['total_timesteps'],
    "callback": training_params['callback']
}

scheduler = ASHAScheduler(
    metric="reward",
    mode="max",
)

# Iterate through the companies and fine-tune the model
for companies in [large_cap_companies]:
    for ticker in companies:
        market_cap_int = get_market_cap(ticker)
        market_cap = metalearner.classify_market_cap(market_cap_int)
        for transaction_size in [1000]:
            scenario = metalearner.classify_scenario(transaction_size)

            finetune_config["train_data"] = train_data
            finetune_config["test_data"] = test_data
            finetune_config["market_cap"] = market_cap
            finetune_config["scenario"] = scenario

            analysis = tune.Tuner(
                macro_trader_tab.fine_tuning_model,
                param_space=finetune_config,
                run_config=train.RunConfig(
                    name=f"{ticker}_{scenario}_{market_cap}",
                    storage_path="~/tune_results",
                ),
                tune_config=tune.TuneConfig(
                    scheduler=scheduler,
                    trial_dirname_creator=lambda trial: trial.trial_id
                )
            )

            results = analysis.fit()
            print(results.get_dataframe())
            # Correct method to use:
            # No need to call metrics_dataframe()
            break
        break

# Best config retrieval (example)
# print("Best config: ", analysis.get_best_config(metric="mean_reward", mode="max"))


2024-09-03 21:04:07,593	INFO worker.py:1616 -- Calling ray.init() again after it has already been called.


Current working directory: /home/ec2-user/SageMaker/ray_results_buy/session_2024-09-03_21-01-11_047433_17532/artifacts/2024-09-03_21-02-15/AAPL_medium-large_large/working_dirs/cc93d_00000


2024-09-03 21:04:07,965	INFO tune.py:616 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949
2024-09-03 21:04:07,966	INFO registry.py:112 -- Detected unknown callable for trainable. Converting to class.


:task_name:bundle_reservation_check_func
:actor_name:ImplicitFunc
:actor_name:fine_tuning_model


<IPython.core.display.HTML object>
:task_name:bundle_reservation_check_func
:actor_name:ImplicitFunc
:actor_name:fine_tuning_model
large cap, large scenario model selected
Model not found at TabModels/model_large_cap_large.h5, training from scratch...


2024-09-03 21:04:08,288	ERROR tune_controller.py:1331 -- Trial task failed for trial fine_tuning_model_0f54c_00000
Traceback (most recent call last):
  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/ray/air/execution/_internal/event_manager.py", line 110, in resolve_future
    result = ray.get(future)
  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/ray/_private/auto_init_hook.py", line 21, in auto_init_wrapper
    return fn(*args, **kwargs)
  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/ray/_private/client_mode_hook.py", line 103, in wrapper
    return func(*args, **kwargs)
  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/ray/_private/worker.py", line 2661, in get
    values, debugger_breakpoint = worker.get_objects(object_refs, timeout=timeout)
  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/ray/_private/worker.py", line 871, in get_objects
    raise va

<IPython.core.display.HTML object>


2024-09-03 21:04:08,348	ERROR tune.py:1037 -- Trials did not complete: [fine_tuning_model_0f54c_00000]
2024-09-03 21:04:08,349	INFO tune.py:1041 -- Total run time: 0.38 seconds (0.32 seconds for the tuning loop).


<IPython.core.display.HTML object>
Empty DataFrame
Columns: []
Index: []


In [ ]:
dfs = {result.path: result.metrics_dataframe for result in results}
[d.mean_accuracy.plot() for d in dfs.values()]

In [5]:
import os
import ray
import pandas as pd
from ray import tune
from ray.tune.schedulers import ASHAScheduler
import json
from collections import defaultdict

# Initialize Ray
ray.init(local_mode=True, num_gpus=1, num_cpus=16, dashboard_host='0.0.0.0', log_to_driver=True, ignore_reinit_error=True)

# Change the current working directory
new_directory =  "/home/ec2-user/SageMaker/Buy_side_model/Blockhouse-ML/"
os.chdir(new_directory)
print("Current working directory:", os.getcwd())  # Correctly print the current directory

# Data directory and other configurations
data_dir = "./Data/"
results = defaultdict(list)

# Define the search space for hyperparameters
search_space = {
    "learning_rate": tune.loguniform(1e-5, 1e-1),
    "n_steps": tune.choice([128, 256, 512]),
    "batch_size": tune.choice([64, 128, 256]),
    "gamma": tune.uniform(0.9, 0.999),
    "clip_range": tune.uniform(0.1, 0.4),
    "n_epochs": tune.choice([4, 6, 8]),
    "ent_coef": tune.loguniform(0.0001, 0.1),
    "resume": False,
    "total_timesteps": 10000,  # Example fixed parameter
    "callback": None  # Example fixed parameter
}

# Scheduler for Ray Tune
scheduler = ASHAScheduler(
    metric="reward",
    mode="max",
)

large_cap_companies = ['AAPL', 'CSCO', 'MCD', 'IBM', 'AMZN', 'TSLA', 'PFE', 'MS','MSFT','NVDA']

# Loop over each ticker
for ticker in tickers:
    # Load the data for the ticker
    file_name = f"processed_data_{ticker}_2024-07-01_2024-07-10.csv"
    file_path = os.path.join(data_dir, file_name)

    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        continue

    processed_data = pd.read_csv(file_path)
    train_data = processed_data[:int(len(processed_data) * 0.8)]
    test_data = processed_data[int(len(processed_data) * 0.8):]

    # Update search space with ticker-specific data
    search_space.update({
        "train_data": train_data,
        "test_data": test_data,
        "market_cap": "large",  # Example value, customize based on your needs
        "scenario": "large"  # Example value, customize based on your needs
    })

    # Create a Tuner object and run the experiment
    tuner = tune.Tuner(
        MacroTraderModel().fine_tuning_model,
        param_space=search_space,
        tune_config=tune.TuneConfig(
            scheduler=scheduler
        )
    )

    print(f"Running tuning for ticker: {ticker}")
    # Run the tuning process
    results = tuner.fit()

    # Save the results to a CSV file
    results_df = results.get_dataframe()
    results_csv_file = f"ray_tune_results_{ticker}.csv"
    results_df.to_csv(results_csv_file, index=False)
    print(f"Results saved to {results_csv_file}")

    # Save the best configuration to a JSON file
    best_result = results.get_best_result(metric="reward", mode="max")
    best_config = best_result.config

    # Ensure the config is JSON serializable
    serializable_config = {key: (value if not isinstance(value, pd.DataFrame) else value.to_dict())
                           for key, value in best_config.items()}

    # Save the serializable configuration
    best_config_file = f"best_config_{ticker}.json"
    with open(best_config_file, 'w') as f:
        json.dump(serializable_config, f, indent=4)

    print(f"Best configuration for {ticker} saved to {best_config_file}")


:job_id:01000000
:task_name:bundle_reservation_check_func
:actor_name:ImplicitFunc
:actor_name:fine_tuning_model
large cap, large scenario model selected


:job_id:01000000
:task_name:bundle_reservation_check_func
:actor_name:ImplicitFunc
:actor_name:fine_tuning_model
2024-09-03 23:42:57,165	ERROR tune_controller.py:1331 -- Trial task failed for trial fine_tuning_model_ec9b0_00000
Traceback (most recent call last):
  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/ray/air/execution/_internal/event_manager.py", line 110, in resolve_future
    result = ray.get(future)
  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/ray/_private/auto_init_hook.py", line 21, in auto_init_wrapper
    return fn(*args, **kwargs)
  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/ray/_private/client_mode_hook.py", line 103, in wrapper
    return func(*args, **kwargs)
  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/ray/_private/worker.py", line 2661, in get
    values, debugger_breakpoint = worker.get_objects(object_refs, timeout=timeout)
  File "/home/ec2-user/a

<IPython.core.display.HTML object>
<IPython.core.display.HTML object>
Results saved to ray_tune_results_AAPL.csv
Best configuration for AAPL saved to best_config_AAPL.json


In [4]:
!pip install polygon
from blockhouse_ml.ml_prod import *

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.9/111.9 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.3/157.3 kB 25.5 MB/s eta 0:00:00


ImportError: cannot import name 'RESTClient' from 'polygon' (/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/polygon/__init__.py)

In [ ]:
ticker = 'AAPL'
tradess = run_pipeline(ticker, start_timestamp = (datetime.today() - timedelta(days=7)).strftime('%Y-%m-%d'), end_timestamp=datetime.today().strftime('%Y-%m-%d'), timeframe=390, inventory=10000, trade_set_counter=1)

In [ ]:
for ticker in large_cap_companies:
    for start_timestamp in 
    tradess = run_pipeline(ticker, start_timestamp = (datetime.today() - timedelta(days=7)).strftime('%Y-%m-%d'), end_timestamp=datetime.today().strftime('%Y-%m-%d'), timeframe=390, inventory=10000, trade_set_counter=1)

In [6]:
!pip install holidays

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 56.5 MB/s eta 0:00:00


In [17]:



import os
import time
from datetime import datetime, timedelta
import pandas as pd
import warnings
import holidays

from blockhouse_ml.utils.macro_model import MetaLearner
from blockhouse_ml.utils.data_handler import DataProcessor, InferenceDataHandler
from blockhouse_ml.utils.macro_model_utils import MacroTraderModel
from blockhouse_ml.utils import fetch_merge_data

os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['OMP_NUM_THREADS'] = '1'

warnings.filterwarnings("ignore")

## Initialize the directories
MACRO_MODEL_DIR = 'TabModels'
DATA_DIR = 'Data'
OUTPUT_DIR = 'MicroInputCSV'
os.makedirs(MACRO_MODEL_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Initialize the models and processors
macro_trader = MacroTraderModel(MACRO_MODEL_DIR)
data_processor = DataProcessor()
inference_data_handler = InferenceDataHandler()
meta = MetaLearner()

def get_schedule(timeframe, transaction_size, market_cap_int, input_row, data):
    """
    Generates a trading schedule based on the transaction size and input data.
    """
    print("LOGGING: Generating Schedule...")
    trades, micro_input = macro_trader.infer_macro(timeframe, transaction_size, market_cap_int, input_row, data, meta=meta, inference_data_handler=inference_data_handler)
    print("Trades Output:", trades)
    print("Micro Input Output:", micro_input)
    print(trades.columns)
    return trades, micro_input




def run_pipeline(ticker, start_timestamp, end_timestamp, timeframe=390, inventory=10000, trade_set_counter=1):
    """
    Runs the pipeline for generating the micro_input DataFrame for a given ticker and date range.
    Returns the generated micro_input DataFrame.
    """
    start_time = time.time()

    # Fetch and process data for the specified ticker and date range
    data = fetch_merge_data.fetch_and_merge_data(ticker, start_date=start_timestamp, end_date=end_timestamp, save_dir=DATA_DIR)
    data = data_processor.add_technical_indicators(data)
    input_row = inference_data_handler.add_forecasts(data)
    market_cap_int = fetch_merge_data.get_market_cap(ticker)
    
    # Generate trades and micro_input
    trades, micro_input = get_schedule(timeframe, inventory, market_cap_int, input_row, data)
    trades.to_csv("tradessss.csv", index=False)
    # Inspect the columns in the trades DataFrame
    print("Columns in 'trades' DataFrame:", trades.columns)
    
    # Assuming 'timestamp' is the correct column name, check if it exists
    if 'timestamp' in trades.columns:
        micro_input['Timestamp'] = trades['timestamp'].tolist()
    else:
        print("Error: 'timestamp' column not found in 'trades' DataFrame")
        return
    
    if isinstance(micro_input['Timestamp'].iloc[0], str) and "Name:" in micro_input['Timestamp'].iloc[0]:
        # Extract just the date and time from the first row
        clean_timestamp = micro_input['Timestamp'].iloc[0].split('Name:')[0].strip()
        micro_input['Timestamp'].iloc[0] = clean_timestamp

    # Ensure the 'Timestamp' column is in proper datetime format for all rows
    micro_input['Timestamp'] = pd.to_datetime(micro_input['Timestamp'], errors='ignore')
    
    # Add additional columns to micro_input
    micro_input['Ticker'] = [ticker] * len(micro_input)
    micro_input['Inventory'] = [inventory] * len(micro_input)
    micro_input['Trade_Set_ID'] = [f"set_{trade_set_counter}"] * len(micro_input)
    trades = pd.DataFrame(trades)
    micro_input['shares'] = trades['shares'].values
    return micro_input, trades



def generate_past_working_days(num_days: int, country='US'):
    """
    Generate a list of date ranges, each covering 1 working day (no weekends or holidays), moving backward from today.

    Args:
    - num_days (int): The number of past working days to generate.
    - country (str): The country for holiday calculations (default is 'US').

    Returns:
    - list: A list of tuples where each tuple is (start_date, end_date), both being the same for 1-day working day ranges.
    """
    today = pd.to_datetime(datetime.today().date())
    date_ranges = []
    us_holidays = holidays.CountryHoliday(country)

    # Continue generating dates until we have the required number of working days
    while len(date_ranges) < num_days:
        # Check if the day is a weekend or a holiday
        if today.weekday() < 5 and today not in us_holidays:
            start_date = today
            end_date = start_date  # For 1-day ranges, end_date is the same as start_date
            date_ranges.append((start_date.strftime('%Y-%m-%d'), end_date.strftime('%Y-%m-%d')))
        # Move to the previous day
        today = today - timedelta(days=1)

    return date_ranges


def run_inference_for_multiple_tickers_and_dates(tickers, date_ranges, timeframe=390, inventory=10000):
    """
    Runs the inference for multiple tickers and multiple date ranges.
    
    Args:
    - tickers (list): List of tickers to process.
    - date_ranges (list): List of tuples containing start and end date strings in the format ('YYYY-MM-DD', 'YYYY-MM-DD').
    - timeframe (int): The timeframe for trade execution (default is 390).
    - inventory (int): The size of the transaction (default is 10000).
    
    Returns:
    - pd.DataFrame: A concatenated DataFrame containing the micro_input for all tickers and date ranges.
    """
    trade_set_counter = 1
    all_data = []

    for ticker in tickers:
        for start_timestamp, end_timestamp in date_ranges:
            print(f"\nProcessing {ticker} from {start_timestamp} to {end_timestamp} (Trade Set {trade_set_counter})")
            # Run the pipeline for each ticker and date range, append the result to all_data
            micro_input, trades = run_pipeline(ticker, start_timestamp, end_timestamp, timeframe, inventory, trade_set_counter)
            
            all_data.append(micro_input)
            
            trade_set_counter += 1

    # Concatenate all micro_input DataFrames into a single DataFrame
    concatenated_data = pd.concat(all_data, ignore_index=True)

    return concatenated_data

# Example usage:

# Define tickers (large-cap companies in your example)
large_cap_companies = ['AAPL' ]#, 'MCD', 'IBM', 'AMZN', 'TSLA', 'PFE', 'MS', 'MSFT', 'NVDA']

# Define date ranges as a list of tuples (start_date, end_date)
date_ranges = generate_past_working_days(250)
# date_ranges = [
#     ('2022-01-01', '2022-01-08'),
#     ('2022-01-09', '2022-01-16'),
#     ('2022-01-17', '2022-01-24')
# ]

# Run inference for multiple tickers and date ranges
all_micro_input_data = run_inference_for_multiple_tickers_and_dates(large_cap_companies, date_ranges)
output_file = 'all_micro_input_data_6.csv'
all_micro_input_data.to_csv(output_file, index=False)
print(f"All micro_input data saved to {output_file}")



Processing AAPL from 2024-09-06 to 2024-09-06 (Trade Set 1)
Merged data saved to merged_data_AAPL_2024-09-06_2024-09-06.csv
LOGGING: Adding Forecasts to data...
LOGGING: Generating Schedule...
large cap, large scenario model selected
loading latest model
low [ 0.05 30.  ]
high[ 0.33000001 50.        ]
------------------------------------------------Class resetted------------------------------------------------
step 35
forecaststep 35


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/arch/univariate/base.py:766: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/arch/univariate/base.py:766: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(


step 45
forecaststep 80


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/arch/univariate/base.py:766: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/arch/univariate/base.py:766: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(


step 44
forecaststep 124


...

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/arch/univariate/base.py:766: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/arch/univariate/base.py:766: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(


step 51
forecaststep 423


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/arch/univariate/base.py:766: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/arch/univariate/base.py:766: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(


Index(['open', 'high', 'low', 'close', 'volume', 'volatility',
       'transaction_cost', 'RSI', 'MACD', 'MACD_signal', 'MACD_hist',
       'Stoch_k', 'Stoch_d', 'OBV', 'Upper_BB', 'Middle_BB', 'Lower_BB',
       'ATR_1', 'ADX', '+DI', '-DI', 'CCI', '5_min_volatility', '5_min_volume',
       '5_min_TC', 'expected_price', 'timestamp', 'forecast_6Hr_open',
       'forecast_6Hr_high', 'forecast_6Hr_low', 'forecast_6Hr_close',
       'forecast_6Hr_volume', 'forecast_6Hr_volatility',
       'forecast_6Hr_transaction_cost'],
      dtype='object')
--------------------------------------------------
Timestamp: 2024-04-01 22:59:00, Action: [ 0.33280001 30.19999981], Shares: 3329, Inventory: 3329, TimeLeft: 359
Timestamp: 2024-04-02 09:30:00, Action: [ 0.0528     50.19999981], Shares: 353, Inventory: 3682, TimeLeft: 308
Timestamp: 2024-04-02 10:21:00, Action: [ 0.0528     42.87600994], Shares: 334, Inventory: 4016, TimeLeft: 265
Timestamp: 2024-04-02 11:04:00, Action: [ 0.15554086 50.19999981], S

KeyError: 'close'